# Centre update rule on the room dataset -- nine runs

Three arms across three seeds:

| arm | what it is |
|---|---|
| `none` | cross-entropy only. **Control**: with seed 42 it must reproduce the existing baseline run (validation 0.5683, test 0.5633). |
| `gradient` | the Centre Loss variant used throughout this thesis |
| `wen` | Algorithm 1 of Wen et al., exactly as published |

Nine runs at roughly two hours each. **Colab will disconnect before this
finishes.** That is expected and handled: every completed arm is written to
Drive, and re-running the training cell skips arms that are already finished
and verified. Just reconnect and run the cells again from the top.

Run the cells in order. Nothing here writes to the folders of any other
experiment.

In [ ]:
# 1. GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2. Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Code -- hard reset, never `git pull`
import os
REPO_DIR = '/content/room-classification'
BRANCH   = 'main'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/SilviuBR24/room-classification.git {REPO_DIR}

# A reused Colab runtime keeps the previous session's clone, and `git pull` can
# leave it behind without reporting a failure -- which once caused a whole run
# to be silently skipped. Resetting to the remote makes the tree match origin
# unconditionally, and the HEAD line below shows what is actually in use.
!cd {REPO_DIR} && git fetch --all --prune && git checkout -B {BRANCH} origin/{BRANCH} && git reset --hard origin/{BRANCH} && git clean -fd
!cd {REPO_DIR} && echo "HEAD is now:" && git log --oneline -1

!pip install -q -r {REPO_DIR}/vit_s16_baseline/requirements.txt

%cd {REPO_DIR}
print('rooms_wen present:', os.path.isdir(f'{REPO_DIR}/rooms_wen'))

In [ ]:
# 4. Dataset -- the same split the 3200 runs used
import os, time
SRC = '/content/drive/MyDrive/Dissertation_Thesis/dataset_split.zip'
DST = '/content/dataset_split'

if os.path.isdir(DST):
    print('already extracted')
else:
    t0 = time.time()
    !unzip -q {SRC} -d /content/
    print(f'extracted in {time.time()-t0:.0f}s')

In [ ]:
# 5. Verify the split before spending eighteen hours on it
from pathlib import Path
ROOT = Path('/content/dataset_split')
EXPECTED = {'train': 19200, 'val': 1800, 'unlabeled': 9000, 'eval': 600}
CLASSES = ['bathroom','bedroom','dining_room','entrance_hall','kitchen','living_room']

ok = True
for part, n in EXPECTED.items():
    got = sum(1 for _ in (ROOT/part).rglob('*.jpg'))
    per = {c: sum(1 for _ in (ROOT/part/c).glob('*.jpg')) for c in CLASSES}
    good = got == n and len(set(per.values())) == 1
    ok &= good
    print(f"{'OK ' if good else 'BAD'} {part:10s} {got:6d} (expected {n})  per class {sorted(set(per.values()))}")

assert ok, 'the split does not match the protocol -- stop and check the archive'
print()
print('Split verified. train 3200/class, val 300/class, test 100/class.')

In [ ]:
# 6. Train and evaluate the nine arms.
#
# Safe to re-run after a disconnect: finished arms are skipped, and an arm is
# only reused if its saved configuration matches the current one.
#
# --only lets you do it in pieces, e.g. --only none  then  --only gradient
!cd /content/room-classification/rooms_wen && python compare_rooms_wen.py     --runs-dir /content/drive/MyDrive/Dissertation_Thesis/dissertation_runs     --num-workers 2

In [ ]:
# 7. Where things stand
import glob, os, pandas as pd
RUNS = '/content/drive/MyDrive/Dissertation_Thesis/dissertation_runs'
csv = os.path.join(RUNS, 'rooms_wen_comparison_results.csv')
if os.path.isfile(csv):
    df = pd.read_csv(csv)
    print(df[['center_mode','seed','best_val_accuracy','test_accuracy','minutes']]
          .to_string(index=False))
    done = df['test_accuracy'].notna().sum()
    print()
    print(f'{done}/9 arms finished')
else:
    print('no summary yet -- the first arm has not finished')

print()
print('run directories on Drive:')
for d in sorted(glob.glob(os.path.join(RUNS, '*rooms_wen_*'))):
    print('  ', os.path.basename(d))

In [ ]:
# 8. Did the control reproduce the existing baseline?
#
# The seed-42 `none` arm and 2026-07-11_17-49_vit_s16_3200_baseline are the same
# configuration trained by two different loops. If they disagree, this folder's
# loop differs somewhere and the comparison is not yet trustworthy.
import glob, os, re

RUNS = '/content/drive/MyDrive/Dissertation_Thesis/dissertation_runs'
def acc(pattern):
    hits = sorted(glob.glob(os.path.join(RUNS, pattern)))
    if not hits: return None, None
    ev = sorted(glob.glob(os.path.join(hits[-1], 'outputs', 'eval_*')))
    if not ev: return hits[-1], None
    txt = open(os.path.join(ev[-1], 'metrics.txt'), encoding='utf-8').read()
    m = re.search(r'Overall accuracy:\s*([0-9.]+)', txt)
    return hits[-1], (float(m.group(1)) if m else None)

_, a = acc('*_vit_s16_3200_baseline')
_, b = acc('*_rooms_wen_none_s42')
print(f'existing baseline (shared loop): {a}')
print(f'control arm  (this loop, s42)  : {b}')
if a is not None and b is not None:
    print(f'difference: {(b-a)*100:+.2f}pp')
    print('The two loops consume randomness differently, so an exact match is not')
    print('expected; a difference of a few points would need explaining.')